# Preparación para perfiles de hospitalización y reglas de asociación

**Período 2023–2025 · Línea de agrupamiento y asociación**

La pregunta de esta línea es qué perfiles de egresos se pueden identificar y qué combinaciones de características se asocian con estadías prolongadas o fallecimientos al egreso. Cada fila representa un egreso hospitalario.

Este notebook parte del Parquet consolidado del equipo. Prepara las variables de la línea, documenta los faltantes y guarda una tabla para el análisis posterior.

## 1. Seleccionar las variables

| Uso | Columnas | Motivo |
| --- | --- | --- |
| Grupo de edad | `GRUPO_EDAD` | Permite distinguir etapas de vida con los intervalos de edad que trae la fuente. |
| Sexo registrado | `SEXO` | Permite caracterizar diferencias demográficas entre perfiles de egresos. |
| Capítulo diagnóstico | `DIAG1_CAPITULO` | Agrupa los diagnósticos principales CIE-10 en conjuntos más amplios y facilita interpretar los perfiles. |
| Resultados de interés | `DIAS_ESTADA`, `CONDICION_EGRESO` | Se usan después de formar los grupos para comparar su duración y condición al egreso. La condición usa 1 para vivo y 2 para fallecido. |
| Procedencia de cada fila | `ANO_FUENTE`, `ID_REGISTRO` | El año identifica el archivo aunque falte el año declarado del egreso. Juntos ubican la fila de origen. Se guardan para auditoría. |

Las entradas del agrupamiento son los tres atributos categóricos de las primeras filas de la tabla. Se usa un solo nivel diagnóstico para evitar repetir la misma información. Si se usa K-means, se convertirán las categorías en columnas indicadoras para evitar que sus códigos internos se interpreten como cantidades.

Abrir este notebook desde la raíz del repositorio y ejecutar las celdas en orden con el entorno de `requirements.txt`.

In [ ]:
from pathlib import Path

import pandas as pd

RUTA_BASE = Path("datos/procesados/egresos_2023_2025_base_comun.parquet")
RESULTADOS = Path("resultados/egresos_perfiles_asociacion")
CLAVE_ORIGEN = ["ANO_FUENTE", "ID_REGISTRO"]
ATRIBUTOS = ["GRUPO_EDAD", "SEXO", "DIAG1_CAPITULO"]
VARIABLES_RESULTADO = ["DIAS_ESTADA", "CONDICION_EGRESO"]
ANIOS_AJUSTE = (2023, 2024)
ANIO_EVALUACION = 2025

if not RUTA_BASE.is_file():
    raise FileNotFoundError(f"Falta la base común en {RUTA_BASE}. Ejecuta el notebook común o recupera el Parquet del equipo.")

perfiles = pd.read_parquet(RUTA_BASE, columns=CLAVE_ORIGEN + ATRIBUTOS + VARIABLES_RESULTADO)

if set(perfiles["ANO_FUENTE"].dropna().unique()) != {*ANIOS_AJUSTE, ANIO_EVALUACION}:
    raise ValueError("La base común debe incluir los archivos de 2023, 2024 y 2025.")
if perfiles[CLAVE_ORIGEN].isna().any().any() or perfiles.duplicated(CLAVE_ORIGEN).any():
    raise ValueError("Cada egreso debe tener una clave única de año de fuente y número de fila.")
if perfiles["DIAS_ESTADA"].isna().any() or not perfiles["DIAS_ESTADA"].gt(0).all():
    raise ValueError("La duración debe estar registrada en días positivos.")
if not perfiles["CONDICION_EGRESO"].isin([1, 2]).all():
    raise ValueError("La condición al egreso debe tener los códigos 1 o 2.")

## 2. Conservar los egresos y marcar los atributos faltantes

La base común ya representa los vacíos y asteriscos como ausentes. Un capítulo diagnóstico ausente puede deberse a que el código no tuvo correspondencia en el diccionario. Se conservan todos los egresos, sus valores faltantes y las estadías extremas.

El análisis principal usará los egresos con los tres atributos disponibles. Si los faltantes se concentran en ciertos grupos, los resultados de los casos completos pueden dar una visión parcial de los egresos. Para comprobar cuánto cambia la interpretación, se compararán los resultados al incluir los faltantes como una categoría diferenciada.

In [ ]:
perfiles[ATRIBUTOS] = perfiles[ATRIBUTOS].astype("category")
disponibles = perfiles[ATRIBUTOS].notna()
perfiles["ATRIBUTOS_COMPLETOS"] = disponibles.all(axis=1)

faltantes = (~disponibles).groupby(perfiles["ANO_FUENTE"]).sum()
faltantes = faltantes.rename(columns={
    "GRUPO_EDAD": "egresos_sin_edad",
    "SEXO": "egresos_sin_sexo",
    "DIAG1_CAPITULO": "egresos_sin_capitulo_diagnostico",
})

## 3. Preparar los resultados para las reglas de asociación

El antecedente de cada regla tendrá uno o más valores de grupo de edad, sexo o capítulo diagnóstico. El consecuente será uno de dos indicadores. `FALLECIMIENTO` identifica la condición 2 y `ESTADIA_PROLONGADA` marca una duración superior al percentil 75 de los egresos completos de 2023–2024.

Los días y la condición originales se conservan para descripción y auditoría. Las reglas usarán sus indicadores derivados como resultados para evitar asociaciones triviales entre un valor y su propia transformación. El umbral se aplica por igual a los tres años y es una definición operativa de este análisis.

Se usarán 2023–2024 para ajustar el agrupamiento y descubrir las reglas. Se reservará 2025 para evaluar su estabilidad temporal. Como el diagnóstico principal se registra al egreso, el estudio es retrospectivo. Las reglas describen asociaciones observadas entre egresos, sin atribuirles una causa.

In [ ]:
ajuste = perfiles["ANO_FUENTE"].isin(ANIOS_AJUSTE) & perfiles["ATRIBUTOS_COMPLETOS"]
estadias_ajuste = perfiles.loc[ajuste, "DIAS_ESTADA"]
if estadias_ajuste.empty:
    raise ValueError("Se necesitan egresos completos de 2023–2024 para calcular el umbral.")

percentil_75 = estadias_ajuste.quantile(0.75)
umbral_dias = int(percentil_75) + 1
perfiles["ESTADIA_PROLONGADA"] = perfiles["DIAS_ESTADA"].ge(umbral_dias)
perfiles["FALLECIMIENTO"] = perfiles["CONDICION_EGRESO"].eq(2)

## 4. Comprobar y guardar la preparación

El resumen muestra cuántos egresos y faltantes hay por año. Los conteos de faltantes de cada atributo pueden coincidir en una misma fila. El total de incompletos cuenta cada egreso una vez. Los conteos de estadías prolongadas y fallecimientos usan todos los egresos de cada año, incluidos los casos incompletos. El Parquet de la línea conserva los egresos y sus indicadores. Ambas salidas se generan localmente.

In [ ]:
resumen = perfiles.groupby("ANO_FUENTE").agg(
    egresos=("ID_REGISTRO", "size"),
    egresos_completos=("ATRIBUTOS_COMPLETOS", "sum"),
    egresos_estadia_prolongada=("ESTADIA_PROLONGADA", "sum"),
    egresos_fallecidos=("FALLECIMIENTO", "sum"),
).join(faltantes)
resumen["egresos_incompletos"] = resumen["egresos"] - resumen["egresos_completos"]
resumen["porcentaje_completos"] = (100 * resumen["egresos_completos"] / resumen["egresos"]).round(2)
resumen["umbral_estadia_dias"] = umbral_dias
if resumen["egresos"].sum() != len(perfiles):
    raise ValueError("El resumen no conserva el número de egresos de la base común.")

RESULTADOS.mkdir(parents=True, exist_ok=True)
ruta_perfiles = RESULTADOS / "egresos_2023_2025_perfiles_preparados.parquet"
temporal = ruta_perfiles.with_suffix(".tmp.parquet")
perfiles.to_parquet(temporal, index=False)
temporal.replace(ruta_perfiles)
resumen.to_csv(RESULTADOS / "resumen_preprocesamiento_perfiles.csv", encoding="utf-8-sig")

incompletos = int(resumen["egresos_incompletos"].sum())
print(f"Se conservaron {len(perfiles):,} egresos. De ellos, {incompletos:,} ({100 * incompletos / len(perfiles):.2f} %) son incompletos.")
print(f"El percentil 75 de 2023–2024 fue {percentil_75:g} días. El umbral es de {umbral_dias} días o más.")
display(resumen)